In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt

In [36]:
df = pd.read_csv('./data_public.csv')
df.head()

,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O,Class
0,231.420023,-12.210984,217.624839,-15.611916,140.047185,76.904999,131.591871,198.160805,82.873279,127.350084,224.592926,-5.992983,-14.689648,143.072058,153.439659,2
1,-38.019270,-14.195695,9.583547,22.293822,-25.578283,-18.373955,-0.094457,-33.711852,-8.356041,23.792402,4.199023,2.809159,-59.330681,-11.685950,1.317104,3
2,-39.197085,-20.418850,21.023083,19.790280,-25.902587,-19.189004,-2.953836,-25.299219,-6.612401,26.285392,5.911292,6.191587,-56.924996,-4.675187,-1.027830,2
3,221.630408,-5.785352,216.725322,-9.900781,126.795177,85.122288,108.857593,197.640135,82.560019,157.105143,212.989231,-3.621070,-15.469156,135.265859,149.212489,2
4,228.558412,-12.447710,204.637218,-13.277704,138.930529,91.101870,115.598954,209.300011,89.961688,130.299732,201.795100,-1.573922,-15.128603,148.368622,147.492663,3


In [37]:
labels = df.columns.tolist()[:-1]
X = pd.DataFrame(data=df.drop('Class', axis=1),
columns=labels)
y = pd.DataFrame(data=df['Class'],
columns=['Class'])
X_train, X_test, y_train, y_test = train_test_split(X,
y,
test_size=0.3)
training_data = pd.concat([X_train,
y_train],
axis=1)
training_data.head()

,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O,Class
33105,-35.719860,-17.347558,7.328224,19.394313,-23.959723,-32.246453,1.381259,-30.427961,-8.197116,25.844963,2.769566,8.668095,-51.755672,-2.083509,-6.855946,2
283413,237.180631,-16.589540,217.516283,-13.438745,129.888383,80.978383,134.192070,200.010701,79.452418,158.691982,211.588745,-9.395300,-9.730429,153.412119,153.017250,2
317901,-43.284137,-12.568157,14.577503,21.848469,-22.759618,-28.875543,4.661535,-27.488896,-6.876554,17.009626,4.810703,-3.700638,-54.312106,-5.636185,3.458398,2
460460,-31.258391,-16.146281,6.492786,14.833837,-24.847844,-33.257478,-0.207201,-23.699937,-7.514042,24.244984,3.443409,2.242625,-54.420729,-1.057446,1.503875,3
14131,-38.236397,-8.983854,9.620306,19.956618,-28.683336,-30.137950,-0.809721,-22.211579,-9.132388,38.215826,2.513197,3.342923,-57.506988,-6.598705,1.434547,2


In [38]:
print(df.isnull().sum())  

num_cols = df.select_dtypes(include='number').columns
imputer_knn = KNNImputer(n_neighbors=5)
df[num_cols] = imputer_knn.fit_transform(df[num_cols])

A        0
B        0
C        0
D        0
E        0
F        0
G        0
H        0
I        0
J        0
K        0
L        0
M        0
N        0
O        0
Class    0
dtype: int64


In [ ]:

mi = mutual_info_classif(X, y.values.ravel())
mi_scores = pd.DataFrame({
    '特征': X.columns, 
    '互信息': mi
}).sort_values('互信息', ascending=False)
zero_mi_cols = mi_scores[mi_scores['互信息'] == 0]['特征'].tolist()
X = X.drop(columns=zero_mi_cols)
df = df.drop(columns=zero_mi_cols)
labels = df.columns.tolist()[:-1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=91)

In [ ]:
# 计算每个特征与 Class 的相关性（Pearson 相关系数）
correlations = X.corrwith(y['Class'])

# 绘制相关性直方图
plt.figure(figsize=(12, 6))
plt.bar(range(len(correlations)), correlations.values)
plt.xlabel('特征索引')
plt.ylabel('与 Class 的 Pearson 相关系数')
plt.title(f'各特征与 Class 的相关性（共 {len(correlations)} 个特征）')
plt.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

# 打印相关性统计摘要
print(f"特征总数: {len(correlations)}")
print(f"正相关特征数: {(correlations > 0).sum()}")
print(f"负相关特征数: {(correlations < 0).sum()}")
print(f"相关性绝对值 > 0.1 的特征数: {(correlations.abs() > 0.1).sum()}")
print(f"\n最大正相关: {correlations.max():.4f}")
print(f"最大负相关: {correlations.min():.4f}")